# Gameplay SLAM with Kuration

This notebook reconstructs one or more gameplay clips with `SLAMReconstructor`, opens the portable result as the normal Kuration `Pipeline`, and selects diverse high-quality windows. From a source checkout, live reconstruction requires the `.[slam]` extra; an existing manifest needs only the base SDK.

In [ ]:
import os
from pathlib import Path

from kuration.gameplay.slam import SLAMReconstructor, input_slam

Set `KURATION_SLAM_VIDEO` to a local gameplay clip. Set `KURATION_SLAM_MODEL` to a Depth Anything 3 model name and pin `KURATION_SLAM_MODEL_REVISION` for reproducible exports.

In [ ]:
video_value = os.environ.get("KURATION_SLAM_VIDEO")
if not video_value:
    raise RuntimeError("Set KURATION_SLAM_VIDEO to a gameplay video before running this notebook")
video = Path(video_value).expanduser().resolve()
if not video.is_file():
    raise FileNotFoundError(video)

model = os.environ.get("KURATION_SLAM_MODEL", "da3-base")
revision = os.environ.get("KURATION_SLAM_MODEL_REVISION", "unpinned")
videos = {video.stem: video}

In [ ]:
slam = SLAMReconstructor(
    depth_model=model,
    depth_model_id=f"{model}@{revision}",
    output_pointcloud=True,
    window_size=48,
    overlap=4,
)
reconstruction = slam.reconstruct(
    videos,
    Path("slam-output"),
    decoder_options={"frame_skip": 4, "color_space": "RGB"},
    resume=True,
)
reconstruction.manifest

In [ ]:
result = (
    reconstruction.to_pipeline(join_video=True)
    .window(32, stride=16)
    .slam_motion_diversity(axis="slam_path_length", bins=8)
    .slam_quality(min_score=0.6)
    .sample("uniform-bins", n=100)
    .collect()
)
len(result.selected)

To curate an existing reconstruction without loading a model, start with `input_slam("slam-output/manifest.json")`. Point-cloud visualization is optional and available through `build_pointcloud_scene()` with the `slam-viz` extra.

In [ ]:
precomputed = input_slam(reconstruction.manifest)
precomputed.describe()